# [Chapter 15 — Vertical Transmission, §15.3] Vertical Transmission and the Modified Reproductive Number

**Book:** *Essential Considerations for Modeling Epidemics* (Hyman/Qu/Xue, 2026)
**Chapter:** Chapter 15 — Vertical Transmission
**Considerations developed:** 4 (force-of-infection direction), 12 (basic vs effective $R$ misapplication)
**Estimated runtime:** ≤ 20 s

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/python/notebooks/chapter_15_vertical_transmission/01_vertical_transmission.ipynb)


## What this notebook does

Many pathogens (HIV, hepatitis B, Chagas, congenital rubella) transmit not only horizontally between individuals but *vertically* from infected mothers to newborns. Vertical transmission adds a second input pathway to the infectious compartment. The notebook shows that the threshold $\mathcal{R}_0$ becomes the sum of horizontal and vertical contributions, and that ignoring the vertical component *underestimates* the threshold and *overestimates* the impact of horizontal interventions (Consideration~12).

## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from shared import book_style, BOOK_COLORS
from shared.seeds import set_seed_chapter_15
from shared.verification import assert_within_tolerance
set_seed_chapter_15()
book_style()


## Modified $SIR_I$ with vertical transmission

$$\dot S = \mu(1 - p_v I) - \beta c_I S I/N - \mu S$$
$$\dot I = \mu p_v I + \beta c_I S I/N - I/\tau_R - \mu I$$
$$\dot R = I/\tau_R - \mu R$$
where $p_v$ is the probability that a newborn from an infected mother is itself infected. The DFE-linearized $\mathcal{R}_0$ becomes
$$\mathcal{R}_0 = \frac{\beta c_I}{1/\tau_R + \mu - \mu p_v}.$$

In [ ]:
from shared.parameters import baseline_chapter_15
p = baseline_chapter_15()

def vt_rhs(y, t, p):
    S, I, R = y
    N = p['N']
    inc_h = p['c_I'] * p['beta'] * S * I / N
    return [p['mu']*(1 - p['p_v']*I) - inc_h - p['mu']*S,
            p['mu']*p['p_v']*I + inc_h - I/p['tau_R'] - p['mu']*I,
            I/p['tau_R'] - p['mu']*R]

p['t_end'] = 365.0 * 30  # 30 years for endemic settling
p['n_points'] = 6001
y0 = [p['S0'], p['I0'], p['R0_init']]
t = np.linspace(p['t_start'], p['t_end'], p['n_points'])
sol = odeint(vt_rhs, y0, t, args=(p,))
S, I, R = sol.T

R0 = (p['c_I']*p['beta']) / (1.0/p['tau_R'] + p['mu'] - p['mu']*p['p_v'])
R0_no_vert = (p['c_I']*p['beta']) / (1.0/p['tau_R'] + p['mu'])
print(f'R_0 with vertical transmission   = {R0:.3f}')
print(f'R_0 without vertical (p_v=0)     = {R0_no_vert:.3f}')
print(f'Ignoring vertical transmission underestimates R_0 by {(R0 - R0_no_vert):.3f}')


## Trajectory

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(t/365, I, color=BOOK_COLORS['infectious'], lw=2)
ax.set_xlabel('time (years)')
ax.set_ylabel('infectious fraction I(t)')
ax.set_title('Vertical-transmission SIR_I, 30-year trajectory')
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()


## Sweeping $p_v$

Hold horizontal $\mathcal{R}_0^{\text{horiz}}$ slightly above 1 (the regime where small additions to the threshold matter most for endemic burden) and sweep $p_v$. Even when vertical transmission's *additive* contribution to $\mathcal{R}_0$ is small in absolute terms, near the threshold it can substantially change the long-run prevalence.

In [ ]:
p2 = dict(p)
# Tune horizontal R_0 to be slightly above threshold
p2['beta'] = p['beta'] * 0.55
R0_h_only = (p2['c_I']*p2['beta']) / (1.0/p2['tau_R'] + p2['mu'])
print(f'Horizontal-only R_0 = {R0_h_only:.4f} (just above threshold)')

p_v_grid = np.linspace(0, 1.0, 21)
I_endemic = []
R0_modified = []
for pv in p_v_grid:
    p2['p_v'] = pv
    sol2 = odeint(vt_rhs, y0, t, args=(p2,))
    I_end = float(sol2[-1, 1])
    R0_mod = (p2['c_I']*p2['beta']) / (1.0/p2['tau_R'] + p2['mu'] - p2['mu']*pv)
    I_endemic.append(I_end)
    R0_modified.append(R0_mod)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].plot(p_v_grid, R0_modified, 'o-', color=BOOK_COLORS['primary'], lw=1.8)
axes[0].set_xlabel('vertical transmission probability $p_v$')
axes[0].set_ylabel('modified $R_0$')
axes[0].set_title('Threshold shift from vertical contribution')
axes[0].grid(True, alpha=0.3)
axes[1].plot(p_v_grid, I_endemic, 'o-', color=BOOK_COLORS['secondary'], lw=1.8)
axes[1].set_xlabel('vertical transmission probability $p_v$')
axes[1].set_ylabel('long-run infectious fraction')
axes[1].set_title('Endemic burden response to $p_v$')
axes[1].grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

# Verify: modified R_0 must monotonically increase with p_v
R0_arr = np.array(R0_modified)
I_arr = np.array(I_endemic)
assert np.all(np.diff(R0_arr) >= -1e-10), 'modified R_0 must be non-decreasing in p_v'
print(f'Modified R_0(p_v=0)   = {R0_arr[0]:.5f}')
print(f'Modified R_0(p_v=1.0) = {R0_arr[-1]:.5f}')
print(f'Endemic I(p_v=0)      = {I_arr[0]:.5f}')
print(f'Endemic I(p_v=1.0)    = {I_arr[-1]:.5f}')
print('\nVerified: vertical transmission shifts the threshold (Consideration 12).')


## What's next

- Chapter 19 (HIV) calibrates this model to perinatal HIV transmission and intervention coverage.
- Mother-to-child transmission interventions (e.g., maternal antiretrovirals) reduce $p_v$ directly; Chapter 10's normalized-sensitivity index for $\mathcal{R}_0$ with respect to $p_v$ tells you how much.
- The principle generalizes to any non-horizontal pathway (e.g., environmental reservoir reseeding).